# Text Summarization

In [ ]:
TEXT = "The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Because it has achieved significance within the past fifty years, Criteria Consideration G applies. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Unlike the Mercury, Gemini, and Apollo programs, the SSP’s emphasis was on cost effectiveness and reusability, and eventually the construction of a space station. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty missions. She had the honor of being chosen as the Return to Flight vehicle after both the Challenger and Columbia accidents. Discovery was the first shuttle to fly with the redesigned SRBs, a result of the Challenger accident, and the first shuttle to fly with the Phase II and Block I SSME. Discovery also carried the Hubble Space Telescope to orbit and performed two of the five servicing missions to the observatory. She flew the first and last dedicated Department of Defense (DoD) missions, as well as the first unclassified defense-related mission. In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle. She was the first orbiter to dock to the ISS, and the first to perform an exchange of a resident crew. Under Criterion C, Discovery is significant as a feat of engineering. According to Wayne Hale, a flight director from Johnson Space Center, the Space Shuttle orbiter represents a “huge technological leap from expendable rockets and capsules to a reusable, winged, hypersonic, cargo-carrying spacecraft.” Although her base structure followed a conventional aircraft design, she used advanced materials that both minimized her weight for cargo-carrying purposes and featured low thermal expansion ratios, which provided a stable base for her Thermal Protection System (TPS) materials. The Space Shuttle orbiter also featured the first reusable TPS; all previous spaceflight vehicles had a single-use, ablative heat shield. Other notable engineering achievements of the orbiter included the first reusable orbital propulsion system, and the first two-fault-tolerant Integrated Avionics System. As Hale stated, the Space Shuttle remains “the largest, fastest, winged hypersonic aircraft in history,” having regularly flown at twenty-five times the speed of sound."

## Extractive Summarization

### nltk

In [ ]:
import nltk, string
from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import WordNetLemmatizer
from heapq import nlargest

In [ ]:
# NLTK ships without data - download once
for pkg in ['punkt_tab', 'stopwords', 'wordnet', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)

def wn_pos(tag):
    """NLTK POS tags -> WordNet tags. Without this the lemmatizer assumes NOUN
       and 'flew'/'flown' never collapse to 'fly'."""
    return {"J": wordnet.ADJ, "V": wordnet.VERB,
            "N": wordnet.NOUN, "R": wordnet.ADV}.get(tag[0], wordnet.NOUN)

def summarize_nltk(text, ratio=0.3):
    stop_words = set(stopwords.words("english"))
    punct = set(string.punctuation) | {"''", "``", "--", "...", "'s"}
    lemm = WordNetLemmatizer()

    sentences = sent_tokenize(text)                    # step 1: split

    # step 2: word frequencies, on POS-aware lemmas
    tagged = nltk.pos_tag(word_tokenize(text))
    freq = {}
    for w, tag in tagged:
        wl = w.lower()
        if wl in stop_words or wl in punct or not any(c.isalnum() for c in wl):
            continue
        key = lemm.lemmatize(wl, wn_pos(tag))
        freq[key] = freq.get(key, 0) + 1

    mx = max(freq.values())                            # step 2b: normalize
    freq = {k: v / mx for k, v in freq.items()}

    # step 3: sentence scores, divided by length
    scores = {}
    for i, sent in enumerate(sentences):
        toks = nltk.pos_tag(word_tokenize(sent))
        content = [(w, t) for w, t in toks
                   if any(c.isalnum() for c in w) and w.lower() not in stop_words]
        if not content:
            continue
        total = sum(freq.get(lemm.lemmatize(w.lower(), wn_pos(t)), 0) for w, t in content)
        scores[i] = total / len(content)               # key by INDEX

    k = max(1, int(len(sentences) * ratio))            # step 4: pick a fraction
    picked = sorted(nlargest(k, scores, key=scores.get))   # step 5: document order

    return " ".join(sentences[i] for i in picked), sentences, picked

### SpaCy

In [ ]:
import spacy
from string import punctuation
from heapq import nlargest



In [ ]:
# the model that splits sentences, tags parts of speech, gives lemmas
nlp = spacy.load("en_core_web_sm")
doc = nlp(TEXT)

In [37]:
sentence_tokens = list(doc.sents)      # 17 sentences, IN DOCUMENT ORDER
sentence_tokens

[The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering.,
 Because it has achieved significance within the past fifty years, Criteria Consideration G applies.,
 Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.,
 Unlike the Mercury, Gemini, and Apollo programs, the SSP’s emphasis was on cost effectiveness and reusability, and eventually the construction of a space station.,
 Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twen

lead-3 = take the first three sentences. That's the whole method.

In [57]:
lead3 = " ".join(s.text.strip() for s in sentence_tokens[:3])
lead3

'The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering. Because it has achieved significance within the past fifty years, Criteria Consideration G applies. Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.'

Why it's the baseline that matters: documents written with the point up front — news, reports, legal descriptions like this one — put the summary in the opening. Look at sentence 0 of your text: "The Orbiter Discovery, OV-103, is considered eligible for listing in the NRHP... under Criterion A... and under Criterion C..." That single sentence contains the document's entire thesis. A frequency algorithm has to work hard to beat "just read the beginning."

In [39]:
stop_words = nlp.Defaults.stop_words
stop_words

{"'d",
 "'ll",
 "'m",
 "'re",
 "'s",
 "'ve",
 'a',
 'about',
 'above',
 'across',
 'after',
 'afterwards',
 'again',
 'against',
 'all',
 'almost',
 'alone',
 'along',
 'already',
 'also',
 'although',
 'always',
 'am',
 'among',
 'amongst',
 'amount',
 'an',
 'and',
 'another',
 'any',
 'anyhow',
 'anyone',
 'anything',
 'anyway',
 'anywhere',
 'are',
 'around',
 'as',
 'at',
 'back',
 'be',
 'became',
 'because',
 'become',
 'becomes',
 'becoming',
 'been',
 'before',
 'beforehand',
 'behind',
 'being',
 'below',
 'beside',
 'besides',
 'between',
 'beyond',
 'both',
 'bottom',
 'but',
 'by',
 'ca',
 'call',
 'can',
 'cannot',
 'could',
 'did',
 'do',
 'does',
 'doing',
 'done',
 'down',
 'due',
 'during',
 'each',
 'eight',
 'either',
 'eleven',
 'else',
 'elsewhere',
 'empty',
 'enough',
 'even',
 'ever',
 'every',
 'everyone',
 'everything',
 'everywhere',
 'except',
 'few',
 'fifteen',
 'fifty',
 'first',
 'five',
 'for',
 'former',
 'formerly',
 'forty',
 'four',
 'from',
 'fron

In [41]:
punct = punctuation + "\n"
punct

'!"#$%&\'()*+,-./:;<=>?@[\\]^_`{|}~\n'

count content words. The premise: important words are repeated words. Strip stop words and punctuation first, or the wins everything.

In [46]:
# --- 1. word frequencies, on LEMMAS, lowercased ---------------------------
#     fixes defect 4: Space/space and mission/missions now merge
word_frequencies = {}
# for each word in the input
for token in doc:
    # skip it if it's a stop word (not related to the concept), punctuation or an empty space
    if token.is_stop or token.is_punct or token.is_space:
        continue

    # use the built-in lemmatization to strip the grammar away from the word
    base_word = token.lemma_.lower()
    # count all base words in the input
    word_frequencies[base_word] = word_frequencies.get(base_word, 0) + 1

word_frequencies

{'orbiter': 9,
 'discovery': 7,
 'ov-103': 1,
 'consider': 1,
 'eligible': 1,
 'list': 1,
 'national': 1,
 'register': 1,
 'historic': 1,
 'places': 1,
 'nrhp': 1,
 'context': 1,
 'u.s.': 2,
 'space': 13,
 'shuttle': 8,
 'program': 4,
 '1969': 1,
 '2011': 1,
 'criterion': 4,
 'area': 2,
 'exploration': 1,
 'transportation': 1,
 'c': 2,
 'engineering': 3,
 'achieve': 1,
 'significance': 1,
 'past': 1,
 'year': 1,
 'criteria': 1,
 'consideration': 1,
 'g': 1,
 'apply': 1,
 'significant': 2,
 'old': 1,
 'extant': 1,
 'vehicle': 3,
 'construct': 1,
 'ssp': 2,
 'long': 1,
 'running': 1,
 'american': 1,
 'date': 1,
 'build': 1,
 'nasa': 1,
 'unlike': 1,
 'mercury': 1,
 'gemini': 1,
 'apollo': 1,
 'emphasis': 1,
 'cost': 1,
 'effectiveness': 1,
 'reusability': 1,
 'eventually': 1,
 'construction': 2,
 'station': 3,
 'include': 2,
 'maiden': 1,
 'voyage': 1,
 'launch': 1,
 'august': 1,
 '30': 1,
 '1984': 1,
 'fly': 8,
 'thirty': 2,
 'time': 2,
 'mission': 5,
 'honor': 1,
 'choose': 1,
 'return

normalize between 0 and 1 for readability

In [47]:
# --- 2. normalize (cosmetic here, but it's the textbook step) -------------
max_freq = max(word_frequencies.values())
word_frequencies = {w: f / max_freq for w, f in word_frequencies.items()}
word_frequencies

{'orbiter': 0.6923076923076923,
 'discovery': 0.5384615384615384,
 'ov-103': 0.07692307692307693,
 'consider': 0.07692307692307693,
 'eligible': 0.07692307692307693,
 'list': 0.07692307692307693,
 'national': 0.07692307692307693,
 'register': 0.07692307692307693,
 'historic': 0.07692307692307693,
 'places': 0.07692307692307693,
 'nrhp': 0.07692307692307693,
 'context': 0.07692307692307693,
 'u.s.': 0.15384615384615385,
 'space': 1.0,
 'shuttle': 0.6153846153846154,
 'program': 0.3076923076923077,
 '1969': 0.07692307692307693,
 '2011': 0.07692307692307693,
 'criterion': 0.3076923076923077,
 'area': 0.15384615384615385,
 'exploration': 0.07692307692307693,
 'transportation': 0.07692307692307693,
 'c': 0.15384615384615385,
 'engineering': 0.23076923076923078,
 'achieve': 0.07692307692307693,
 'significance': 0.07692307692307693,
 'past': 0.07692307692307693,
 'year': 0.07692307692307693,
 'criteria': 0.07692307692307693,
 'consideration': 0.07692307692307693,
 'g': 0.07692307692307693,
 '

score each sentence by summing its words' frequencies. This is the actual "which sentences matter" decision.

In [48]:
# --- 3. sentence scores, DIVIDED BY LENGTH -------------------------------
#     fixes defect 5: stops long sentences winning automatically
sentence_tokens = list(doc.sents)
sentence_scores = {}
for sent in sentence_tokens:
    content = [t for t in sent if not t.is_punct and not t.is_space]
    if not content:
        continue
    total = sum(word_frequencies.get(t.lemma_.lower(), 0) for t in content)
    sentence_scores[sent] = total / len(content)

sentence_scores

{The Orbiter Discovery, OV-103, is considered eligible for listing in the National Register of Historic Places (NRHP) in the context of the U.S. Space Shuttle Program (1969-2011) under Criterion A in the areas of Space Exploration and Transportation and under Criterion C in the area of Engineering.: 0.1394230769230769,
 Because it has achieved significance within the past fifty years, Criteria Consideration G applies.: 0.04395604395604396,
 Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.: 0.16730769230769227,
 Unlike the Mercury, Gemini, and Apollo programs, the SSP’s emphasis was on cost effectiveness and reusability, and eventually the construction of a space station.: 0.10153846153846154,
 Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, m

pick the top N. This is the summarization. nlargest(N, sentence_scores, key=sentence_scores.get) returns the N highest-scoring sentences.

In [49]:
# --- 4. select a FRACTION, not everything --------------------------------
#     fixes defect 1
select_length = max(1, int(len(sentence_tokens) * 0.3))
selected = nlargest(select_length, sentence_scores, key=sentence_scores.get)
selected

[In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle.,
 The Space Shuttle orbiter also featured the first reusable TPS; all previous spaceflight vehicles had a single-use, ablative heat shield.,
 According to Wayne Hale, a flight director from Johnson Space Center, the Space Shuttle orbiter represents a “huge technological leap from expendable rockets and capsules to a reusable, winged, hypersonic, cargo-carrying spacecraft.”,
 Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.,
 Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly 

Step 4 returns sentences ranked by score: [0, 9, 12, 2, 4]. Joined that way, the summary jumps from the intro to the ISS paragraph and back. Unreadable.

In [ ]:
# --- 5. restore DOCUMENT order -------------------------------------------
#     fixes defect 3
selected = sorted(selected, key=lambda s: sentence_tokens.index(s)) # looks up "where was this one originally"


In [54]:
selected

[In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle.,
 The Space Shuttle orbiter also featured the first reusable TPS; all previous spaceflight vehicles had a single-use, ablative heat shield.,
 According to Wayne Hale, a flight director from Johnson Space Center, the Space Shuttle orbiter represents a “huge technological leap from expendable rockets and capsules to a reusable, winged, hypersonic, cargo-carrying spacecraft.”,
 Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA.,
 Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly 

In [56]:
summary = " ".join(s.text.strip() for s in selected)
print(summary)
print(f"\n{len(sentence_tokens)} sentences -> {len(selected)}")
print(f"compression: {len(summary)/len(TEXT):.1%}")

In addition, Discovery was vital to the construction of the International Space Station (ISS); she flew thirteen of the thirty-seven total missions flown to the station by a U.S. Space Shuttle. The Space Shuttle orbiter also featured the first reusable TPS; all previous spaceflight vehicles had a single-use, ablative heat shield. According to Wayne Hale, a flight director from Johnson Space Center, the Space Shuttle orbiter represents a “huge technological leap from expendable rockets and capsules to a reusable, winged, hypersonic, cargo-carrying spacecraft.” Under Criterion A, Discovery is significant as the oldest of the three extant orbiter vehicles constructed for the Space Shuttle Program (SSP), the longest running American space program to date; she was the third of five orbiters built by NASA. Including her maiden voyage (launched August 30, 1984), Discovery flew to space thirty-nine times, more than any of the other four orbiters; she was also the first orbiter to fly twenty mi

## Abstractive summarization

In [ ]:
from transformers import pipeline

# stage 1 - your existing extractive summarizer, but keep MORE than final
extracted, _, _ = summarize_nltk(TEXT, ratio=0.5)     # ~50% instead of 30%

# stage 2 - abstractive rewrite
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6")
result = summarizer(extracted, max_length=120, min_length=40, do_sample=False)
print(result[0]["summary_text"])